In [ ]:
import datetime
import pandas as pd
from scipy.io import loadmat
import numpy as np
import os
from pandas import DataFrame
from load_data_function import load_data,save_data
import re
from load_data_function import fig_plot,battery_soh_plot,smooth_soh

In [ ]:
print("Input which battery you need to extract data from. Choose from the following")
print("Battery Number: B0005,B0006,B0007,B0018")

#define a function for extracting discharge and charge data
def disch_data(battery):
  mat = loadmat('D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data/NASA_dataset/1. BatteryAgingARC-FY08Q4/' + battery + '.mat') #get the .mat file
  print('Total data in dataset: ', len(mat[battery][0, 0]['cycle'][0])) #get the length of the data from number of cycles
  c = 0 #set a variable to zero
  disdataset = [] #create an empty list for discharge data
  capacity_data = []

  for i in range(len(mat[battery][0, 0]['cycle'][0])):
    row = mat[battery][0, 0]['cycle'][0, i] #get each row of the cycle
    if row['type'][0] == 'discharge': #if the row is a dicharge cycle
      ambient_temperature = row['ambient_temperature'][0][0] #get temp,date_time stamp,capacity,voltage,current etc,.
      date_time = datetime.datetime(int(row['time'][0][0]),
                               int(row['time'][0][1]),
                               int(row['time'][0][2]),
                               int(row['time'][0][3]),
                               int(row['time'][0][4])) + datetime.timedelta(seconds=int(row['time'][0][5]))
      data = row['data']
      capacity = data[0][0]['Capacity'][0][0]
      for j in range(len(data[0][0]['Voltage_measured'][0])):
        voltage_measured = data[0][0]['Voltage_measured'][0][j]
        current_measured = data[0][0]['Current_measured'][0][j]
        temperature_measured = data[0][0]['Temperature_measured'][0][j]
        current_load = data[0][0]['Current_load'][0][j]
        voltage_load = data[0][0]['Voltage_load'][0][j]
        time = data[0][0]['Time'][0][j]
        disdataset.append([c + 1, ambient_temperature, date_time, capacity,
                        voltage_measured, current_measured,
                        temperature_measured, current_load,
                        voltage_load, time])
        capacity_data.append([c + 1, ambient_temperature, date_time, capacity])
      c = c + 1
  print(disdataset[0])
  return [pd.DataFrame(data=disdataset,
                       columns=['cycle', 'ambient_temperature', 'datetime',
                                'capacity', 'voltage_measured',
                                'current_measured', 'temperature_measured',
                                'current', 'voltage', 'time']),
          pd.DataFrame(data=capacity_data,
                       columns=['cycle', 'ambient_temperature', 'datetime',
                                'capacity'])]

def charge_data(battery): #similarly write a fn for charge data
  mat = loadmat('D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data/NASA_dataset/1. BatteryAgingARC-FY08Q4/' + battery + '.mat') #get the .mat file
  c = 0
  chdataset = []

  for i in range(len(mat[battery][0, 0]['cycle'][0])):
    row = mat[battery][0, 0]['cycle'][0, i]
    if row['type'][0] == 'charge' :

      ambient_temperature = row['ambient_temperature'][0][0]
      date_time = datetime.datetime(int(row['time'][0][0]),
                               int(row['time'][0][1]),
                               int(row['time'][0][2]),
                               int(row['time'][0][3]),
                               int(row['time'][0][4])) + datetime.timedelta(seconds=int(row['time'][0][5]))
      data = row['data']
      for j in range(len(data[0][0]['Voltage_measured'][0])):
        voltage_measured = data[0][0]['Voltage_measured'][0][j]
        current_measured = data[0][0]['Current_measured'][0][j]
        temperature_measured = data[0][0]['Temperature_measured'][0][j]
        current_charge = data[0][0]['Current_charge'][0][j]
        voltage_charge = data[0][0]['Voltage_charge'][0][j]
        time = data[0][0]['Time'][0][j]
        chdataset.append([c + 1, ambient_temperature, date_time,
                        voltage_measured, current_measured,
                        temperature_measured, current_charge,
                        voltage_charge, time])
      c = c + 1
  print(chdataset[788])
  return chdataset

B = input()
chdataset = charge_data(B)
chdf=pd.DataFrame(data=chdataset,columns=['cycle', 'ambient_temperature', 'datetime',
                                'voltage_measured','current_measured',
                                'temperature_measured','current',
                                'voltage', 'time'])
pd.set_option('display.max_columns', 10)
chdf

disdf,capacity = disch_data(B)
pd.set_option('display.max_columns', 10)
disdf

cycling_data = pd.concat([chdf, disdf])
cycling_data=cycling_data.sort_values(['cycle'], ascending=[True])
cycling_data



In [ ]:
os.makedirs('D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data/NASA_dataset/data', exist_ok=True)
cycling_data.to_csv( 'D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data/NASA_dataset/data/'+ "NASA_cycle_dataset_" + B + ".csv")

In [ ]:
NASA_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data/NASA_dataset/data'
package_list=['package_1']
NASA_data={}
NASA_SOH={}
for i,package in enumerate(package_list):

    print(f'package: {package}')
    battery_list=os.listdir(NASA_path)

    print(battery_list)

    package1={}
    package2={}
    for j,battery in enumerate(battery_list):
        print(f'battery: {battery}')
        battery_path=os.path.join(NASA_path,battery)
        battery_data = pd.read_csv(battery_path)

        print(battery_data.shape)

        voltage=[]
        current=[]
        time=[]
        capacity=[]
        package1[f'battery_{j+1}']=[]
        package2[f'battery_{j+1}']=[]
        cycle_num= battery_data['cycle'].unique()
        cycle_num=sorted(cycle_num)
        #print(cycle_num)
        for k in cycle_num:
            #print(f'cycle: {k}')
            cycle_data=battery_data[battery_data['cycle']==k]
            voltage=cycle_data['voltage_measured'].values.reshape(1,-1)

            current=cycle_data['current_measured'].values.reshape(1,-1)
            time_segment = cycle_data['time'].values.reshape(1,-1)
            time=time_segment  # 转换为秒
            #time=time.reshape(1,-1)
            discharge_capacity=cycle_data['capacity'].values.reshape(1,-1)
            discharge_capacity=np.float32(discharge_capacity)
            #charge_capacity=np.float32(charge_capacity)
            #capacity=np.concatenate((discharge_capacity,charge_capacity),axis=1)
            #print(discharge_capacity.shape)
            capacity_max=np.max(discharge_capacity)
            soh=capacity_max/2.0
            print(soh)
            package1[f'battery_{j+1}'].append(np.concatenate((voltage,current,time),axis=0))
            package2[f'battery_{j+1}'].append(soh)
    NASA_data[f'package_{i+1}']=package1
    NASA_SOH[f'package_{i+1}']=package2

In [ ]:
print(NASA_data.keys())
print(NASA_SOH.keys())
print(NASA_data['package_1'].keys())
print(NASA_data['package_1']['battery_1'][0].shape)
print(NASA_data['package_1']['battery_1'][0][0])

In [ ]:
package='package_1'
battery_soh_plot(NASA_SOH,NASA_SOH[package].keys(),package=package)

In [ ]:
for package in NASA_data.keys():
    for battery in NASA_data[package].keys():
        if len(NASA_data[package][battery])!= len(NASA_SOH[package][battery]):
            print(f'battery {battery} has different length of data and SOH,data length: {len(NASA_data[package][battery])}, SOH length: {len(NASA_SOH[package][battery])}')

In [ ]:
save_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data/NASA_dataset'
os.makedirs(save_path, exist_ok=True)
#save_data(NASA_data,os.path.join(save_path,'NASA_data.pkl'))
#save_data(NASA_SOH,os.path.join(save_path,'NASA_SOH.pkl'))
NASA_data=load_data(os.path.join(save_path,'NASA_data.pkl'))
NASA_SOH=load_data(os.path.join(save_path,'NASA_SOH.pkl'))

In [ ]:
fig_plot(NASA_data['package_1']['battery_1'][0][1])